### Template

```python
# Document A: FAQ
strategy_A = "custom (question-answer pair)"  - Custom splitting by Q:/A: pairs
reason_A = "FAQs are naturally structured as independent question-answer pairs. Each Q&A is a self-contained unit of information that directly answers a specific user query. Chunking by individual Q&A pairs preserves semantic completeness, avoids splitting meaningful units, and ensures that retrieved chunks are directly relevant in a RAG system."
chunks_A = [
    "Q: What is the return policy?\nA: Items can be returned within 30 days of purchase with original receipt.",
    "Q: Do you offer international shipping?\nA: Yes, we ship to over 50 countries worldwide. Shipping times vary by location.",
    "Q: How do I track my order?\nA: Use the tracking number sent to your email after shipment."
]

# Document B: Technical Documentation
strategy_B = "custom (step-based)"  - Custom splitting by numbered steps
reason_B = "Technical documentation, especially installation guides, is organized around sequential steps. Each step (or closely related sub-lines within a step) forms a coherent, actionable unit. Splitting by steps keeps instructions intact and prevents breaking procedural flow, which is critical for accurate retrieval and user understanding."
chunks_B = [
    "Step 1: Download the installer from our website.\nExtract the zip file to your desired location.",
    "Step 2: Run setup.exe as administrator.\nFollow the on-screen instructions.",
    "Step 3: Configure your API key in the settings file.\nThe settings file is located at config/settings.json."
]

# Document C: Article
strategy_C = "paragraph"  - Splitting by natural paragraph breaks
reason_C = "Articles consist of continuous prose where each paragraph typically represents a single coherent idea or topic. Paragraph-level chunking maintains semantic integrity better than sentence-level (which can feel fragmented) or larger sections (which may mix multiple ideas). It provides a good balance between chunk size and contextual completeness for narrative or expository text."
chunks_C = [
    "The Future of Renewable Energy\n\nSolar and wind power have seen tremendous growth in recent years. As technology improves\nand costs decrease, renewable energy becomes increasingly competitive with fossil fuels.",
    "Energy storage solutions are critical for renewable adoption. Battery technology advances\nenable better grid management and reliability. This addresses the intermittent nature of\nsolar and wind power.",
    "Policy support and public awareness continue to drive the transition. Many countries have\nset ambitious renewable energy targets for the coming decades."
]
```

In [1]:
import re
from typing import List

In [3]:
# Document A: FAQ - Custom Q&A pair splitting
document_a = """
Q: What is the return policy?
A: Items can be returned within 30 days of purchase with original receipt.

Q: Do you offer international shipping?
A: Yes, we ship to over 50 countries worldwide. Shipping times vary by location.

Q: How do I track my order?
A: Use the tracking number sent to your email after shipment.
"""

def chunk_faq(text: str) -> List[str]:
    """
    Splits FAQ text into individual Question-Answer pairs.
    Assumes format: Q: ... A: ...
    Handles optional blank lines between pairs.
    """
    # Normalize line endings and strip leading/trailing whitespace
    text = text.strip()
    
    # Split on 'Q:' which marks the start of a new pair
    # Use positive lookbehind to keep 'Q:' in the chunk
    pairs = re.split(r'(?=\n?Q:)', text)
    
    # Clean up each pair
    chunks = []
    for pair in pairs:
        pair = pair.strip()
        if pair:  # Skip empty strings
            chunks.append(pair)
    
    return chunks

chunks_A = chunk_faq(document_a)

print("=== FAQ Chunks ===")
for i, chunk in enumerate(chunks_A, 1):
    print(f"Chunk {i}:\n{chunk}\n")

=== FAQ Chunks ===
Chunk 1:
Q: What is the return policy?
A: Items can be returned within 30 days of purchase with original receipt.

Chunk 2:
Q: Do you offer international shipping?
A: Yes, we ship to over 50 countries worldwide. Shipping times vary by location.

Chunk 3:
Q: How do I track my order?
A: Use the tracking number sent to your email after shipment.



In [5]:
# Document B: Technical Documentation - Custom step-based splitting
document_b = """
Installation Guide

Step 1: Download the installer from our website.
Extract the zip file to your desired location.

Step 2: Run setup.exe as administrator.
Follow the on-screen instructions.

Step 3: Configure your API key in the settings file.
The settings file is located at config/settings.json.
"""

def chunk_technical_steps(text: str) -> List[str]:
    """
    Splits technical/procedural docs by numbered steps (Step 1:, Step 2:, etc.).
    Includes all lines that belong to that step until the next step begins.
    """
    # Normalize and split into lines
    lines = [line.rstrip() for line in text.strip().split('\n')]
    
    chunks = []
    current_chunk = []
    
    for line in lines:
        # Detect start of a new step
        if re.match(r'^Step \d+:', line.strip()):
            if current_chunk:
                chunks.append('\n'.join(current_chunk).strip())
                current_chunk = []
            current_chunk.append(line)
        else:
            # Skip empty lines at the very beginning (like titles)
            if current_chunk or line.strip():
                current_chunk.append(line)
    
    # Add the last chunk
    if current_chunk:
        chunks.append('\n'.join(current_chunk).strip())
    
    return chunks

chunks_B = chunk_technical_steps(document_b)

print("\n=== Technical Documentation Chunks ===")
for i, chunk in enumerate(chunks_B, 1):
    print(f"Chunk {i}:\n{chunk}\n")


=== Technical Documentation Chunks ===
Chunk 1:
Installation Guide

Chunk 2:
Step 1: Download the installer from our website.
Extract the zip file to your desired location.

Chunk 3:
Step 2: Run setup.exe as administrator.
Follow the on-screen instructions.

Chunk 4:
Step 3: Configure your API key in the settings file.
The settings file is located at config/settings.json.



In [9]:
# Document C: Article - Paragraph-level splitting

document_c = """
The Future of Renewable Energy

Solar and wind power have seen tremendous growth in recent years. As technology improves
and costs decrease, renewable energy becomes increasingly competitive with fossil fuels.

Energy storage solutions are critical for renewable adoption. Battery technology advances
enable better grid management and reliability. This addresses the intermittent nature of
solar and wind power.

Policy support and public awareness continue to drive the transition. Many countries have
set ambitious renewable energy targets for the coming decades.
"""

def chunk_by_paragraph(text: str) -> List[str]:
    """
    Splits narrative text (articles, reports) by natural paragraph breaks.
    Handles paragraphs separated by one or more blank lines.
    Preserves internal line breaks if present (e.g., title).
    """
    # Split on one or more blank lines
    paragraphs = re.split(r'\n\s*\n', text.strip())
    
    # Clean up each paragraph
    chunks = [para.strip() for para in paragraphs if para.strip()]
    
    return chunks

chunks_C = chunk_by_paragraph(document_c)

print("\n=== Article Chunks ===")
for i, chunk in enumerate(chunks_C, 1):
    print(f"Chunk {i}:\n{chunk}\n")


=== Article Chunks ===
Chunk 1:
The Future of Renewable Energy

Chunk 2:
Solar and wind power have seen tremendous growth in recent years. As technology improves
and costs decrease, renewable energy becomes increasingly competitive with fossil fuels.

Chunk 3:
Energy storage solutions are critical for renewable adoption. Battery technology advances
enable better grid management and reliability. This addresses the intermittent nature of
solar and wind power.

Chunk 4:
Policy support and public awareness continue to drive the transition. Many countries have
set ambitious renewable energy targets for the coming decades.

